# 11.5 HTTP - From Sockets to requests

**Prerequisites:** 11.2 TCP Client and Server, 08 File Handling (JSON)  
**Target:** Python 3.12+ (notes flag 3.13/3.14 differences)

### What you'll learn
- HTTP as it really is: **text over a TCP socket**
- Writing a request by hand and reading the raw response
- 🔴 How HTTP solves the framing problem from 11.2 - twice
- Running a local server with `http.server`
- `urllib.request` (stdlib) vs `requests` (the one everyone uses)
- 🔴 **Always pass a timeout** - and what happens when you do not
- `Session`, connection reuse, and retries with backoff
- Status codes, `raise_for_status()`, and JSON APIs
- What HTTPS adds, and why you never implement it yourself

---

## HTTP is simpler than it looks

Every web page, every REST API, every `pip install` is this: open a TCP socket, send some **text**, read some text back. You already know how to do that from **11.2**.

A request:

```
GET /status HTTP/1.1\r\n          <- method, path, version
Host: example.com\r\n             <- headers, one per line
User-Agent: demo/1.0\r\n
\r\n                              <- BLANK LINE ends the headers
```

A response:

```
HTTP/1.1 200 OK\r\n               <- version, status code, reason
Content-Type: application/json\r\n
Content-Length: 27\r\n            <- how many BODY bytes follow
\r\n                              <- blank line again
{"status": "ok", "jobs": 3}      <- exactly 27 bytes
```

🔴 **Line endings are `\r\n`, not `\n`.** Carriage-return line-feed, and the spec means it. A lone `\n` is the classic hand-rolled-HTTP bug.

> **Note how HTTP frames itself** — the problem from **11.2**. Headers are **newline-delimited** and end with a blank line; then `Content-Length` says exactly how many body bytes follow, a **length prefix**. Both techniques, in one protocol. HTTP had the same problem you did, and solved it the same two ways.

### A server to talk to

Everything in this notebook runs against a local `http.server` on port 0 — no internet, no rate limits, nothing to break. It runs on a daemon thread, exactly like **11.2**.

`ThreadingHTTPServer` gives each request its own thread, which is the thread-per-client pattern from **11.4** already built into the standard library.

In [ ]:
import http.server
import json
import sys
import threading
import time

REQUEST_LOG = []
ABANDONED = []          # clients that hung up before we replied


class DemoHandler(http.server.BaseHTTPRequestHandler):
    protocol_version = "HTTP/1.1"          # enables keep-alive; needs Content-Length

    def log_message(self, *args):
        """Silence the default stderr logging - we keep our own list."""


    def _respond(self, status, payload, *, delay=0.0):
        if delay:
            time.sleep(delay)
        body = json.dumps(payload).encode("utf-8")
        self.send_response(status)
        self.send_header("Content-Type", "application/json")
        self.send_header("Content-Length", str(len(body)))   # the length prefix
        self.end_headers()
        self.wfile.write(body)

    def do_GET(self):
        REQUEST_LOG.append(("GET", self.path))
        if self.path == "/status":
            self._respond(200, {"status": "ok", "jobs": 3})
        elif self.path == "/slow":
            self._respond(200, {"status": "eventually"}, delay=3.0)
        elif self.path == "/boom":
            self._respond(500, {"error": "internal"})
        else:
            self._respond(404, {"error": "not found", "path": self.path})

    def do_POST(self):
        length = int(self.headers.get("Content-Length", 0))
        raw = self.rfile.read(length)          # read EXACTLY that many bytes
        REQUEST_LOG.append(("POST", self.path))
        try:
            sent = json.loads(raw) if raw else {}
        except json.JSONDecodeError:
            self._respond(400, {"error": "invalid JSON"})
            return
        self._respond(201, {"created": True, "echo": sent})


class QuietServer(http.server.ThreadingHTTPServer):
    """A client hanging up mid-response is normal, not a crash.

    The timeout demonstration further down abandons a request on purpose.
    socketserver would otherwise print a full traceback from the handler
    thread. Note this belongs on the SERVER: ThreadingMixIn calls
    self.handle_error() on the server object, not on the request handler.
    """

    def handle_error(self, request, client_address):
        exc = sys.exception()
        if isinstance(exc, (ConnectionAbortedError, ConnectionResetError,
                            BrokenPipeError)):
            ABANDONED.append(type(exc).__name__)
            return
        super().handle_error(request, client_address)


httpd = QuietServer(("127.0.0.1", 0), DemoHandler)
HOST, PORT = httpd.server_address
BASE = f"http://{HOST}:{PORT}"

server_thread = threading.Thread(target=httpd.serve_forever, daemon=True)
server_thread.start()
print("local server running at", BASE)

## Speaking HTTP by hand

No library. A socket, some bytes, and the protocol from the top of this notebook. This is worth doing once — after it, HTTP stops being magic.

Note `Connection: close`, which tells the server to close after replying. Without it, HTTP/1.1 keeps the connection open for reuse and our `recv` loop would sit waiting for data that never comes.

In [ ]:
import socket

request = (
    "GET /status HTTP/1.1\r\n"
    f"Host: {HOST}:{PORT}\r\n"
    "User-Agent: hand-written/1.0\r\n"
    "Accept: application/json\r\n"
    "Connection: close\r\n"
    "\r\n"
).encode("ascii")

print("--- what we send ---")
print(request.decode().replace("\r\n", "\\r\\n\n"))

with socket.create_connection((HOST, PORT), timeout=5.0) as sock:
    sock.sendall(request)
    chunks = []
    while True:
        chunk = sock.recv(4096)
        if not chunk:                      # b'' -> server closed (11.2)
            break
        chunks.append(chunk)
raw = b"".join(chunks)

print("--- what comes back, byte for byte ---")
print(repr(raw))

# Split headers from body on the blank line
head, _, body = raw.partition(b"\r\n\r\n")
print("\n--- headers ---")
for line in head.decode().split("\r\n"):
    print("   ", line)
print("\n--- body ---")
print("   ", body.decode())
print("\n    parsed:", json.loads(body))

### 🔴 Why you should not keep doing this

That worked, and it is nowhere near enough for real use. A correct HTTP client also handles:

- **Chunked transfer encoding**, when the server does not know the length in advance (so there is no `Content-Length` to rely on)
- Redirects (`301`, `302`, `307`) and redirect loops
- `gzip`/`deflate`/`br` content encodings
- Cookies, authentication, proxies
- Connection reuse and pooling
- Character-set detection from headers *and* body
- TLS certificate verification
- Header injection defences, duplicate headers, case-insensitive lookup

That is why `requests` exists. Write it by hand once to understand it; then use a library, always.

## `urllib.request` - the standard library

In the stdlib, so it needs no install — worth knowing for scripts that must run anywhere, and for environments where you cannot add dependencies.

The API is clunky: HTTP errors arrive as **exceptions** rather than as a response you can inspect, and you must decode bytes yourself.

In [ ]:
import urllib.error
import urllib.request

req = urllib.request.Request(
    f"{BASE}/status",
    headers={"Accept": "application/json", "User-Agent": "urllib-demo/1.0"},
)

with urllib.request.urlopen(req, timeout=5) as response:   # 🔴 always a timeout
    print("status  :", response.status, response.reason)
    print("headers :", dict(response.headers))
    payload = json.loads(response.read().decode("utf-8"))
    print("body    :", payload)

print("\n-- an error status arrives as an EXCEPTION --")
try:
    with urllib.request.urlopen(f"{BASE}/boom", timeout=5) as response:
        print("unexpected success")
except urllib.error.HTTPError as exc:
    print(f"  HTTPError {exc.code}: {exc.reason}")
    print("  body still readable:", exc.read().decode())
    # 🔴 HTTPError is not just an exception - it IS the response object,
    # file-like and backed by a temporary file. Unclosed, it raises
    # ResourceWarning when collected.
    exc.close()
    print("  ^ 500 is a perfectly valid HTTP response, but urllib raises on it")
except urllib.error.URLError as exc:
    print("  URLError (could not connect):", exc.reason)

## `requests` - what almost everyone uses

Not in the standard library (`pip install requests`), and the de facto standard anyway. Same operations, far less ceremony.

| | `urllib.request` | `requests` |
|---|---|---|
| GET JSON | build `Request`, `urlopen`, `read`, `decode`, `json.loads` | `requests.get(url).json()` |
| 404 / 500 | raises `HTTPError` | returns a response; you decide |
| Query strings | build by hand with `urlencode` | `params={...}` |
| POST JSON | encode, set headers manually | `json={...}` |
| Connection reuse | manual | `Session` |

🔴 **`requests` does not raise on 404 or 500.** A response *is* the result. Call `raise_for_status()` when you want an exception — forgetting it means happily parsing an error page as if it were your data.

In [ ]:
try:
    import requests
except ImportError:
    requests = None
    print("requests is not installed - pip install requests")

if requests is not None:
    response = requests.get(f"{BASE}/status", timeout=5)   # 🔴 always a timeout
    print("status_code :", response.status_code)
    print("ok          :", response.ok)
    print("json()      :", response.json())
    print("headers     :", response.headers["Content-Type"])
    print("elapsed     :", f"{response.elapsed.total_seconds():.3f}s")

    print("\n-- an error status is NOT an exception --")
    bad = requests.get(f"{BASE}/boom", timeout=5)
    print("  status_code:", bad.status_code)
    print("  ok         :", bad.ok)
    print("  json()     :", bad.json(), " <- parsed happily")
    print("\n  🔴 Nothing raised. Without raise_for_status() you would carry on")
    print("     as though this were real data.")

    try:
        bad.raise_for_status()
    except requests.HTTPError as exc:
        print("\n  raise_for_status() ->", type(exc).__name__)
        print("   ", str(exc)[:78])

In [ ]:
if requests is not None:
    print("-- query parameters, encoded for you --")
    response = requests.get(f"{BASE}/search",
                            params={"q": "socket programming", "page": 2},
                            timeout=5)
    print("  URL built:", response.url)
    print("  status   :", response.status_code, "(our server only knows a few paths)")

    print("\n-- POST JSON --")
    created = requests.post(
        f"{BASE}/jobs",
        json={"name": "reindex-search", "retries": 2},   # sets Content-Type for you
        timeout=5,
    )
    print("  status:", created.status_code)
    print("  body  :", created.json())

    print("\n-- what the server saw --")
    for method, path in REQUEST_LOG[-3:]:
        print(f"   {method:<5} {path}")

## 🔴 Timeouts are not optional

**`requests` has no default timeout.** Neither does `urllib`. Omit it and a request can hang **forever** — not for 30 seconds, forever — if the server accepts your connection and then says nothing.

This is the single most common cause of a Python service that mysteriously stops doing anything: every worker thread parked on a `requests.get()` that will never return.

```
    requests.get(url, timeout=5)          one number: applies to BOTH phases
    requests.get(url, timeout=(3, 10))    (connect timeout, read timeout)
```

The tuple form is usually what you want: fail fast if you cannot *reach* the server, but allow a slow response once connected.

> ⚠️ The read timeout is **between bytes**, not a total deadline. A server dribbling one byte every second keeps the connection alive indefinitely without ever tripping it.

In [ ]:
if requests is not None:
    print("our /slow endpoint sleeps 3s before replying\n")

    started = time.perf_counter()
    try:
        requests.get(f"{BASE}/slow", timeout=1.0)
        print("  completed (unexpected)")
    except requests.Timeout as exc:
        print(f"  timeout=1.0 -> gave up after {time.perf_counter() - started:.2f}s")
        print("   ", type(exc).__name__)

    started = time.perf_counter()
    response = requests.get(f"{BASE}/slow", timeout=10.0)
    print(f"\n  timeout=10.0 -> got {response.status_code} "
          f"after {time.perf_counter() - started:.2f}s")

    print("\n  With no timeout at all, the first call would simply never return")
    print("  if the server never replied. There is no safety net.")

    print("\n  meanwhile, on the server side:")
    print("    clients that hung up before we replied:", ABANDONED or "none yet")
    print("    ^ the abandoned request finished its 3s of work and then found\n"
          "      nobody listening. A timeout stops the CLIENT waiting; it does\n"
          "      not stop the SERVER working.")

## `Session`: connection reuse and retries

Every `requests.get()` opens a **new TCP connection** — and with HTTPS, a new TLS handshake, which is several round trips. Hitting the same host repeatedly, that is the dominant cost.

A `Session` keeps a connection pool and reuses connections. It also carries shared headers, cookies and auth, and is where you mount a **retry policy**.

```
    Retry(total=3, backoff_factor=0.5, status_forcelist=[502, 503, 504])
                   ^^^^^^^^^^^^^^^^^^  waits 0.5s, 1s, 2s - exponential backoff
```

🔴 **Retry only what is safe to repeat.** `GET` is idempotent; `POST` may not be — retrying one can create two orders. `allowed_methods` controls this, and by default `POST` is excluded for exactly this reason.

In [ ]:
if requests is not None:
    from requests.adapters import HTTPAdapter
    from urllib3.util.retry import Retry

    retry = Retry(
        total=3,
        backoff_factor=0.3,                       # 0.3s, 0.6s, 1.2s
        status_forcelist=[502, 503, 504],         # retry these; NOT 500 or 404
        allowed_methods={"GET", "HEAD", "OPTIONS"},   # idempotent only
    )

    with requests.Session() as session:
        session.headers.update({"User-Agent": "notes-demo/1.0"})
        session.mount("http://", HTTPAdapter(max_retries=retry))

        started = time.perf_counter()
        for _ in range(5):
            session.get(f"{BASE}/status", timeout=5)
        pooled = time.perf_counter() - started

    started = time.perf_counter()
    for _ in range(5):
        requests.get(f"{BASE}/status", timeout=5)
    fresh = time.perf_counter() - started

    print(f"5 requests with a Session : {pooled * 1000:.1f} ms")
    print(f"5 separate requests.get() : {fresh * 1000:.1f} ms")
    print("\n  On loopback the gap is small - there is no real latency and no TLS.")
    print("  Against a remote HTTPS host the Session is dramatically faster,")
    print("  because it skips a TCP handshake AND a TLS handshake each time.")

## Status codes worth knowing

| Range | Meaning | Common ones |
|---|---|---|
| **2xx** | success | `200 OK`, `201 Created`, `204 No Content` |
| **3xx** | redirect | `301` permanent, `302`/`307` temporary, `304 Not Modified` |
| **4xx** | **you** made a mistake | `400`, `401` unauthenticated, `403` forbidden, `404`, `429` rate limited |
| **5xx** | **the server** made a mistake | `500`, `502` bad gateway, `503` unavailable, `504` gateway timeout |

The 4xx/5xx split decides your response to failure:

- **4xx — do not retry.** The request is wrong; sending it again changes nothing. (`429` is the exception: retry, after the delay in `Retry-After`.)
- **5xx — retrying may work.** Something transient broke. Back off exponentially.

🔴 `401 Unauthorized` actually means *unauthenticated* — you have not proven who you are. `403 Forbidden` means you have, and still may not. The names have been wrong since 1997.

## HTTPS, briefly

HTTPS is HTTP through a **TLS** layer that adds encryption, integrity and identity — the last being the point most people forget. Certificate verification is what stops you talking to an impostor; without it, encryption alone buys you very little.

In Python this is `ssl.wrap_socket`-style plumbing, and `requests` does all of it for you, verifying certificates **by default**.

```
    requests.get(url, verify=False)      🔴 NEVER in production.
                     ^^^^^^^^^^^^^^      Disables the identity check entirely.
```

If you hit a certificate error, fix the certificate chain or point `verify=` at the right CA bundle. Turning verification off converts a loud error into a silent vulnerability — and it is a *very* common piece of bad advice online.

> This notebook runs plain HTTP on loopback deliberately: no certificate to manage, and nothing leaves the machine. Real services should be HTTPS-only.

In [ ]:
import ssl

context = ssl.create_default_context()
print("default TLS context:")
print("  verify mode      :", context.verify_mode.name,
      "- certificates are checked")
print("  check_hostname   :", context.check_hostname,
      "- the name must match the certificate")
print("  minimum version  :", context.minimum_version.name,
      "- older, broken protocols refused")
print("  CA certs loaded  :", len(context.get_ca_certs()))

print("\nand what verify=False would leave you with:")
unsafe = ssl.create_default_context()
unsafe.check_hostname = False                  # order matters: hostname first
unsafe.verify_mode = ssl.CERT_NONE
print("  verify mode      :", unsafe.verify_mode.name)
print("  check_hostname   :", unsafe.check_hostname)
print("\n  Still encrypted - but with no idea who is on the other end,")
print("  which is most of what TLS was for.")

In [ ]:
# ---- tidy up ----
httpd.shutdown()          # stops serve_forever
httpd.server_close()      # releases the listening socket
server_thread.join(timeout=5)

print("server stopped:", not server_thread.is_alive())
print(f"requests handled: {len(REQUEST_LOG)}")

leftover = [t for t in threading.enumerate() if t is not threading.main_thread()]
print("\nthreads still alive:", [t.name for t in leftover] or "none")

if leftover:
    print("\nThese are per-request handler threads still holding HTTP")
    print("keep-alive connections open - protocol_version = 'HTTP/1.1' means")
    print("the server does NOT close after replying, so they sit waiting for")
    print("a follow-up request that never comes.")
    print("\n  all of them are daemon threads:", all(t.daemon for t in leftover))
    print("  ^ ThreadingHTTPServer sets daemon_threads, so they cannot keep")
    print("    the interpreter alive. Sending 'Connection: close', or closing")
    print("    the Session, would retire them immediately.")

## Where next

This folder covered the transport. **20 Working with APIs** builds on it: REST conventions, authentication, pagination, rate limiting, and structuring a client properly.

| You learned here | Used there |
|---|---|
| `Session`, retries, timeouts | the backbone of any API client |
| Status codes and `raise_for_status` | error handling policy |
| JSON request and response bodies | every REST API |

---

## Common Mistakes & Pitfalls

1. 🔴 **Omitting `timeout=`.** `requests` and `urllib` both wait forever by default. This is the classic cause of a service that silently stops responding.
2. 🔴 **Assuming `requests` raises on 404 or 500.** It does not. Call `raise_for_status()`, or check `response.ok` yourself.
3. 🔴 **`verify=False` to make a certificate error go away.** That disables identity checking and makes the connection impersonatable.
4. **Using `\n` instead of `\r\n` in hand-written HTTP.** The spec requires CRLF.
5. **Parsing HTTP with `split()` on the raw bytes.** Chunked encoding, duplicate headers and continuation lines will defeat you. Use a library.
6. **A new connection for every request to the same host.** Use a `Session`; on HTTPS you are paying for a TLS handshake every time.
7. **Retrying `POST` blindly.** It may not be idempotent - two retries can mean three orders. Retry idempotent methods only.
8. **Retrying 4xx.** The request is wrong; repeating it will not help. `429` is the exception, and honour `Retry-After`.
9. **Trusting `Content-Length` from an untrusted server.** A lying length is how a naive client is made to hang or over-allocate.

## Best Practices

- Pass `timeout=` to every single request. Prefer the `(connect, read)` tuple.
- Call `raise_for_status()` unless you are deliberately inspecting the status.
- Use a `Session` for more than one request to the same host.
- Configure retries with exponential backoff, restricted to idempotent methods and 5xx/429.
- Let the library build query strings (`params=`) and bodies (`json=`) - never concatenate them.
- Keep certificate verification on. Fix the trust chain instead of disabling it.
- Set a descriptive `User-Agent`; it is what an operator sees when your client misbehaves.
- Test clients against a local `http.server`, as here - fast, offline, and you can make it fail on demand.

## Practice Exercises

Try these before moving on.

1. Add a `/redirect` endpoint returning `302` with a `Location` header. Does `requests` follow it by default? What does `response.history` contain?
2. Write the hand-rolled client again, but parse the response with `http.client.HTTPResponse` instead of splitting bytes. Which edge cases does that handle for you?
3. 🔴 Make the server accept a connection and never reply. Call it with no timeout and a hard time limit on the cell. Then add `timeout=2` and compare.
4. Add a `/flaky` endpoint failing with `503` twice before succeeding. Configure `Retry` so a `Session` gets through, and log each attempt.
5. Implement `Retry-After` handling: on `429`, read the header and sleep that long before retrying.
6. POST invalid JSON to `/jobs` and confirm the `400` path. Then make the client distinguish a 4xx (do not retry) from a 5xx (retry).
7. Compare `Session` and plain `requests.get` against a real HTTPS host, 20 requests each. Why is the gap so much larger than on loopback?